# FedFlower Phase 2 — Federated Learning

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

This notebook:
- Warm-starts from `best_model.pth` (upload when prompted)
- Runs a **5-client** FedAvg simulation (408 imgs/client, 10 rounds)
- Runs a **10-client** FedAvg simulation (204 imgs/client, 10 rounds)
- Tracks cross-entropy loss **per client per round**
- Reports per-client accuracy on local data and global test set
- Reports per-client macro F1 on global test set
- Produces `federated_accuracy.png`, `federated_loss.png`, `federated_results.json`

## Cell 1 — Install & Imports

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install scikit-learn matplotlib numpy -q
print('✅ Libraries ready')

import torch, torch.nn as nn, torch.optim as optim
import torchvision.datasets as datasets, torchvision.transforms as transforms, torchvision.models as models
from torch.utils.data import DataLoader, Subset, ConcatDataset
import numpy as np, matplotlib.pyplot as plt, copy, json, os
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if not torch.cuda.is_available():
    print('⚠️  No GPU — go to Runtime → Change Runtime Type → T4 GPU')

✅ Libraries ready
Device: cuda


## Cell 2 — Upload best_model.pth

When the file picker appears, select `best_model.pth` from your laptop (downloaded after Phase 1).

In [ ]:
from google.colab import files
print('Upload final_model_fulltrain.pth from your laptop...')
uploaded = files.upload()
assert 'final_model_fulltrain.pth' in uploaded, ' Wrong filename — must be final_model_fulltrain.pth'
print(f' Uploaded final_model_fulltrain.pth ({os.path.getsize("final_model_fulltrain.pth")/1e6:.1f} MB)')

Upload final_model_fulltrain.pth from your laptop...


Saving final_model_fulltrain.pth to final_model_fulltrain.pth
 Uploaded final_model_fulltrain.pth (98.8 MB)


## Cell 3 — Define FlowerCNN & Load Dataset

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4), nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.backbone(x)

# Strong augmentation for federated training data (per CONTEXT.md)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Clean transform for val/test — must match inference preprocessing exactly
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.Flowers102('./data', split='train', download=True, transform=train_transform)
val_data   = datasets.Flowers102('./data', split='val',   download=True, transform=train_transform)
test_data  = datasets.Flowers102('./data', split='test',  download=True, transform=test_transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print(f'   Federated pool: {len(train_data)+len(val_data)} images (train+val)')

100%|██████████| 345M/345M [00:18<00:00, 18.4MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.22MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 42.7MB/s]

✅ Train: 1020 | Val: 1020 | Test: 6149
   Federated pool: 2040 images (train+val)


## Cell 4 — Helper Functions

`local_train` returns **(state_dict, avg_loss)** so we can track cross-entropy per client per round.

In [ ]:
def local_train(global_model, loader, local_epochs=3, lr=5e-5, device='cuda'):
    """Train a local copy; return (weights, avg_cross_entropy_loss)."""
    local_model = copy.deepcopy(global_model)
    local_model.train()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, local_model.parameters()), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    total_loss, total_batches = 0.0, 0
    for _ in range(local_epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(local_model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            total_batches += 1
    avg_loss = total_loss / total_batches if total_batches > 0 else 0.0
    return local_model.state_dict(), avg_loss


def fedavg(global_model, client_weights_list, client_sizes):
    """Weighted average of client weights by dataset size."""
    total = sum(client_sizes)
    avg_w = copy.deepcopy(client_weights_list[0])
    for key in avg_w:
        avg_w[key] = torch.zeros_like(avg_w[key], dtype=torch.float32)
        for i, cw in enumerate(client_weights_list):
            avg_w[key] += cw[key].float() * (client_sizes[i] / total)
    global_model.load_state_dict(avg_w)
    return global_model


def evaluate_accuracy(model, loader, device):
    """Return Top-1 accuracy (%) on a DataLoader."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


def evaluate_accuracy_and_f1(model, loader, device):
    """Return (accuracy %, macro F1) on a DataLoader."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = 100.0 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return acc, f1


print('✅ local_train, fedavg, evaluate_accuracy, evaluate_accuracy_and_f1 defined')

✅ local_train, fedavg, evaluate_accuracy, evaluate_accuracy_and_f1 defined


## Cell 5 — Run 5-Client Federated Training (10 Rounds)

Each of the 5 clients gets ~408 images. Expected time: ~50 min.

## Cell 6 — Run 10-Client Federated Training (10 Rounds)

Each of the 10 clients gets ~204 images. Expected time: ~50 min.

## Cell 7 — Per-Client Accuracy & F1 After Training

For the **5-client** model: each client is evaluated on (a) their own local data and (b) the full 6,149-image test set. Macro F1 is reported on the global test set.

## Cell 8 — Plot 1: Federated Accuracy vs Rounds

5-client line, 10-client line, and centralized 89.04% dashed baseline.

## Cell 9 — Plot 2: Cross-Entropy Loss per Round

## Cell 10 — Save federated_results.json & Download All Files

Download these three files and keep them alongside `best_model.pth` on your laptop.

Cell 11 — Multiple seeds (5-client + 10-client × seeds 42/43/44)

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def run_federated_iid(num_clients, seed, warmstart_path, global_rounds=10, local_epochs=3):
    set_seed(seed)
    indices = np.random.permutation(len(all_data))
    client_idx = np.array_split(indices, num_clients)
    loaders = [DataLoader(Subset(all_data, idx.tolist()), batch_size=32, shuffle=True, num_workers=2)
               for idx in client_idx]

    model = FlowerCNN(num_classes=102).to(device)
    model.load_state_dict(torch.load(warmstart_path, map_location=device))

    for rnd in range(global_rounds):
        weights, sizes = [], []
        for loader in loaders:
            w, _ = local_train(model, loader, local_epochs, lr=5e-5, device=device)
            weights.append(w); sizes.append(len(loader.dataset))
        model = fedavg(model, weights, sizes)

    final_acc, final_f1 = evaluate_accuracy_and_f1(model, test_loader, device)
    return final_acc, final_f1

WARMSTART = 'final_model_fulltrain.pth'
SEEDS = [42, 43, 44]
seed_results_fulltrain = {5: [], 10: []}

for num_clients in [5, 10]:
    for seed in SEEDS:
        print(f'\n=== {num_clients}-client | seed={seed} | warm-start={WARMSTART} ===')
        acc, f1 = run_federated_iid(num_clients, seed, WARMSTART)
        seed_results_fulltrain[num_clients].append({'seed': seed, 'final_acc': acc, 'final_f1': f1})
        print(f'seed={seed} → acc={acc:.2f}%  f1={f1:.4f}')

        # save after EVERY run, not just at the end — protects against disconnects
        with open('seed_results_fulltrain.json', 'w') as f:
            json.dump(seed_results_fulltrain, f, indent=2)

print('\n' + '=' * 50)
for nc in [5, 10]:
    accs = [r['final_acc'] for r in seed_results_fulltrain[nc]]
    print(f'{nc}-client: mean={np.mean(accs):.2f}%  std={np.std(accs):.2f}%')

print('\n✅ Saved seed_results_fulltrain.json (updated after every run)')


=== 5-client | seed=42 | warm-start=final_model_fulltrain.pth ===
seed=42 → acc=94.05%  f1=0.9391

=== 5-client | seed=43 | warm-start=final_model_fulltrain.pth ===
seed=43 → acc=93.93%  f1=0.9380

=== 5-client | seed=44 | warm-start=final_model_fulltrain.pth ===
seed=44 → acc=93.95%  f1=0.9365

=== 10-client | seed=42 | warm-start=final_model_fulltrain.pth ===
seed=42 → acc=93.87%  f1=0.9358

=== 10-client | seed=43 | warm-start=final_model_fulltrain.pth ===
seed=43 → acc=93.72%  f1=0.9354

=== 10-client | seed=44 | warm-start=final_model_fulltrain.pth ===
seed=44 → acc=93.79%  f1=0.9355

5-client: mean=93.98%  std=0.05%
10-client: mean=93.79%  std=0.06%

✅ Saved seed_results_fulltrain.json (updated after every run)


Cell 12 — Non-IID (Dirichlet, 5-client, alpha=0.5)


In [ ]:
from collections import Counter

def dirichlet_partition(num_classes, num_clients, alpha, labels, seed):
    set_seed(seed)
    client_idx = [[] for _ in range(num_clients)]
    labels = np.array(labels)
    for c in range(num_classes):
        idx_c = np.where(labels == c)[0]
        np.random.shuffle(idx_c)
        proportions = np.random.dirichlet(alpha * np.ones(num_clients))
        cuts = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        for i, part in enumerate(np.split(idx_c, cuts)):
            client_idx[i].extend(part.tolist())
    return client_idx

labels_all = list(train_data._labels) + list(val_data._labels)  # matches all_data order
client_idx_noniid = dirichlet_partition(102, 5, 0.5, labels_all, seed=42)

loaders_noniid = [DataLoader(Subset(all_data, idx), batch_size=32, shuffle=True, num_workers=2)
                  for idx in client_idx_noniid]

per_client_counts = {}
for i, idx in enumerate(client_idx_noniid):
    counts = Counter([labels_all[j] for j in idx])
    per_client_counts[f'client_{i+1}'] = dict(counts)
    print(f'Client {i+1}: {len(idx)} images, {len(counts)} unique species, top 5: {counts.most_common(5)}')

model_noniid = FlowerCNN(num_classes=102).to(device)
model_noniid.load_state_dict(torch.load('final_model_fulltrain.pth', map_location=device))

for rnd in range(10):
    weights, sizes = [], []
    for loader in loaders_noniid:
        w, _ = local_train(model_noniid, loader, 3, lr=5e-5, device=device)
        weights.append(w); sizes.append(len(loader.dataset))
    model_noniid = fedavg(model_noniid, weights, sizes)
    acc = evaluate_accuracy(model_noniid, test_loader, device)
    print(f'Round {rnd+1}: test_acc={acc:.2f}%')

final_acc, final_f1 = evaluate_accuracy_and_f1(model_noniid, test_loader, device)
print(f'\n✅ Non-IID (5-client, alpha=0.5) — Acc: {final_acc:.2f}%  F1: {final_f1:.4f}')

with open('noniid_results.json', 'w') as f:
    json.dump({'final_acc': final_acc, 'final_f1': final_f1,
                'per_client_species_counts': per_client_counts}, f, indent=2)
print('✅ Saved noniid_results.json')

Client 1: 381 images, 64 unique species, top 5: [(27, 16), (64, 16), (34, 15), (52, 14), (26, 13)]
Client 2: 393 images, 76 unique species, top 5: [(68, 16), (74, 16), (95, 16), (46, 14), (1, 13)]
Client 3: 398 images, 84 unique species, top 5: [(33, 19), (22, 18), (91, 18), (41, 17), (82, 16)]
Client 4: 404 images, 74 unique species, top 5: [(2, 17), (70, 17), (83, 17), (37, 15), (100, 15)]
Client 5: 464 images, 102 unique species, top 5: [(35, 18), (58, 17), (44, 16), (87, 16), (19, 14)]
Round 1: test_acc=93.69%
Round 2: test_acc=93.61%
Round 3: test_acc=93.66%
Round 4: test_acc=93.85%
Round 5: test_acc=93.76%
Round 6: test_acc=93.74%
Round 7: test_acc=93.79%
Round 8: test_acc=93.77%
Round 9: test_acc=93.67%
Round 10: test_acc=93.89%

✅ Non-IID (5-client, alpha=0.5) — Acc: 93.89%  F1: 0.9382
✅ Saved noniid_results.json
